# Plasma-column WarpX method comparison

This notebook runs and compares the first two methods directly: seeded neutralization and Python-callback proton-impact source. The C++ BackgroundMCC method requires applying the patch and rebuilding WarpX, so this notebook provides the run commands for the patched executable/script.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

WORK = Path.home() / "Work" / "simulation_codes-working"
WARPX_DATA_DIR = WORK / "warpx-data"

SEEDED_SCRIPT = Path("plasma_column_mcc_picmi_v6.py").resolve()
CALLBACK_SCRIPT = Path("plasma_column_callback_source_picmi.py").resolve()

os.environ["WARPX_DATA_DIR"] = str(WARPX_DATA_DIR)
os.environ["LD_LIBRARY_PATH"] = "/home/cspark/Work/simulation_codes-working/warpx/install/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

print("Python:", sys.executable)
print("WarpX data:", WARPX_DATA_DIR)
print("Seeded script:", SEEDED_SCRIPT)
print("Callback script:", CALLBACK_SCRIPT)


## 1. Check pywarpx and proton-impact ionization files

In [ ]:
subprocess.run([sys.executable, "-c", "from pywarpx import picmi; print('pywarpx OK')"], check=True, env=os.environ.copy())

for gas in ["H2", "Kr"]:
    p = WARPX_DATA_DIR / "MCC_cross_sections" / gas / "proton_impact_ionization.dat"
    print(gas, p, "OK" if p.exists() else "MISSING")


## 2. Run seeded neutralization cases

In [ ]:
common = [
    "--warpx_data_dir", str(WARPX_DATA_DIR),
    "--pressure_torr", "1e-5",
    "--plasma_age", "2e-4",
    "--max_steps", "500",
    "--diag_period", "100",
    "--reduced_diag_period", "1",
    "--nx", "24", "--ny", "24", "--nz", "128",
]

seeded_cases = [
    ("seeded_vacuum", ["--gas", "H2", "--neutralization", "0.0", "--mcc", "none"]),
    ("seeded_H2", ["--gas", "H2", "--neutralization", "-1", "--mcc", "none"]),
    ("seeded_Kr", ["--gas", "Kr", "--neutralization", "-1", "--mcc", "none"]),
]

for name, args in seeded_cases:
    out = Path("runs") / name
    cmd = [sys.executable, str(SEEDED_SCRIPT), "--run", "--output_dir", str(out)] + common + args
    print("\nRUN:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=os.environ.copy())


## 3. Run Python-callback source cases

In [ ]:
callback_common = [
    "--warpx_data_dir", str(WARPX_DATA_DIR),
    "--pressure_torr", "1e-5",
    "--max_steps", "500",
    "--diag_period", "100",
    "--nx", "24", "--ny", "24", "--nz", "128",
    "--source_every_n_steps", "1",
    "--source_weight_min", "1.0",
]

callback_cases = [
    ("callback_vacuum", ["--gas", "H2", "--enable_ionization_source", "0"]),
    ("callback_H2", ["--gas", "H2", "--enable_ionization_source", "1"]),
    ("callback_Kr", ["--gas", "Kr", "--enable_ionization_source", "1"]),
]

for name, args in callback_cases:
    out = Path("runs") / name
    cmd = [sys.executable, str(CALLBACK_SCRIPT), "--run", "--output_dir", str(out)] + callback_common + args
    print("\nRUN:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=os.environ.copy())


## 4. Compare method outputs

Seeded runs write `neutralization_from_particle_number.csv` and `neutralization_model.csv`. Callback runs write full diagnostics and `neutralization_source_model.csv`; the raw `ParticleNumber` reduced diagnostic can be inspected under `reducedfiles/`.

In [ ]:
def maybe_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print("Missing:", path)
    return None

seeded_hist = {}
seeded_model = {}
for name, _ in seeded_cases:
    out = Path("runs") / name
    seeded_hist[name] = maybe_read_csv(out / "neutralization_from_particle_number.csv")
    seeded_model[name] = maybe_read_csv(out / "neutralization_model.csv")

callback_model = {}
for name, _ in callback_cases:
    out = Path("runs") / name
    callback_model[name] = maybe_read_csv(out / "neutralization_source_model.csv")

plt.figure()
for name, df in seeded_model.items():
    if df is not None:
        plt.plot(df["step"], df["neutralization_fraction_model"], "--", label=f"{name}: seeded analytic")
for name, df in callback_model.items():
    if df is not None:
        plt.plot(df["step"], df["source_integral_fraction_1_minus_exp"], label=f"{name}: source integral")
plt.xlabel("PIC step")
plt.ylabel("source/neutralization model fraction")
plt.legend()
plt.grid(True)
plt.show()


## 5. C++ BackgroundMCC ion-impact ionization method

Apply and build the C++ patch first. Then run a case using the input snippets in `warpx_ion_impact_ionization_patch.zip`. The comparison quantities are the same: \(N_e(t)\), \(N_i(t)\), \(N_p(t)\), beam envelope, emittance, and transmission.

In [ ]:
print("C++ method checklist:")
print("1. Install linear cross-section files in warpx-data/MCC_cross_sections/H2 and Kr.")
print("2. Unzip warpx_ion_impact_ionization_patch.zip.")
print("3. Copy IonImpactIonization.H into WarpX Source/Particles/Collision/BackgroundMCC/.")
print("4. Merge INTEGRATION_SNIPPETS.cpp into the relevant WarpX source files.")
print("5. Rebuild WarpX.")
print("6. Add inputs_snippet_H2 or inputs_snippet_Kr to your input deck.")
